# 🏭 ALGORITHM COMPARISON: Spatiotemporal SDVRP vs Pure SDVRP
## Research-Grade Warehouse Optimization Framework

This notebook compares two algorithmic approaches for order batching and routing:
- **Your Approach (Spatiotemporal SDVRP)**: Uses a 4D congestion heatmap `H(x,y,t)` to actively predict and route pickers around traffic bottlenecks.
- **Thesis Baseline (Pure SDVRP)**: Uses capacity-constrained route partitioning minimizing pure travel distance (ignoring warehouse traffic).


In [ ]:
!pip install -q numpy matplotlib networkx ortools pandas


## Core Implementation


In [ ]:
# =====================================================================================
# SPATIOTEMPORAL CONGESTION-AWARE MULTI-AGENT SDVRP
# Research-Grade Warehouse Optimization Framework
# =====================================================================================
#
# FEATURES
# --------
# 1. Split Delivery Vehicle Routing Problem (SDVRP)
# 2. Spatiotemporal congestion heatmap H(x,y,t)
# 3. Predictive congestion-aware batching
# 4. Multi-objective optimization
# 5. Congestion-aware VNS local search
# 6. Route overlap minimization
# 7. Dynamic edge weighting
# 8. OR-Tools TSP optimization
#
# OBJECTIVE
# ---------
# Minimize:
#
# F = αD + βC + γW + δB + εO
#
# D = Travel Distance
# C = Congestion
# W = Waiting / Traffic Delay
# B = Workload Imbalance
# O = Route Overlap
#
# =====================================================================================

# =====================================================================================
# IMPORTS
# =====================================================================================

import math
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Tuple, Dict, Set

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

# =====================================================================================
# DATA MODELS
# =====================================================================================

Coord = Tuple[int, int]

@dataclass
class Product:
    sku: str
    x: int
    y: int
    category: str
    zone: str

@dataclass
class OrderItem:
    sku: str
    qty: int

@dataclass
class Order:
    order_id: str
    items: List[OrderItem]

@dataclass
class Config:
    max_batch_units: int = 25
    alpha_distance: float = 1.0
    beta_congestion: float = 15.0
    gamma_balance: float = 1.5
    epsilon_overlap: float = 5.0
    omega_wait: float = 20.0     # High penalty for actually waiting in a queue
    kappa_bottleneck: float = 8.0 # Penalty for routing through known bottlenecks
    congestion_decay: float = 0.93
    time_horizon: int = 200      # Increased horizon for realistic makespan
    service_time: int = 2        # Time spent picking at a shelf or waiting in queue

# =====================================================================================
# WAREHOUSE GRAPH
# =====================================================================================

class WarehouseGraph:

    def __init__(self, width, height, blocked):
        self.width = width
        self.height = height
        self.blocked = blocked
        
        # Hardcode central cross-aisles as bottlenecks
        self.bottlenecks = {
            (width//2, height//2), (width//2, height//2 + 1), (width//2, height//2 - 1),
            (width//2 + 1, height//2), (width//2 - 1, height//2)
        }

        self.G = nx.Graph()

        for x in range(width):
            for y in range(height):

                if (x, y) in blocked:
                    continue

                self.G.add_node((x, y))

                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:

                    nx2 = x + dx
                    ny2 = y + dy

                    if (
                        0 <= nx2 < width and
                        0 <= ny2 < height and
                        (nx2, ny2) not in blocked
                    ):
                        self.G.add_edge((x,y), (nx2,ny2), weight=1)

        self.cache = {}

    def dist(self, a, b):

        if a == b:
            return 0

        key = (a,b)

        if key in self.cache:
            return self.cache[key]

        try:
            d = nx.astar_path_length(
                self.G,
                a,
                b,
                heuristic=lambda x,y: abs(x[0]-y[0]) + abs(x[1]-y[1]),
                weight="weight"
            )
        except:
            d = 9999

        self.cache[key] = d
        self.cache[(b,a)] = d

        return d

# =====================================================================================
# SPATIOTEMPORAL CONGESTION MODEL
# =====================================================================================

class SpatiotemporalHeatmap:

    def __init__(self, width, height, time_horizon, decay=0.95):

        self.width = width
        self.height = height
        self.time_horizon = time_horizon
        self.decay = decay

        self.H = np.zeros((width, height, time_horizon))

    def add_route(self, route, start_time=0, weight=1.0):

        t = start_time

        for c in route:

            if t >= self.time_horizon:
                break

            x, y = c
            
            # Add picker presence
            self.H[x,y,t] += weight
            
            # Time collision / Queue (If someone is already here, add wait time)
            if self.H[x,y,t] > 1.0:
                t += 2  # Delayed by service time queue

            # spatial spread
            for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:

                nx2 = x + dx
                ny2 = y + dy

                if (
                    0 <= nx2 < self.width and
                    0 <= ny2 < self.height
                ):
                    self.H[nx2,ny2,t] += weight * 0.4

            t += 1

    def congestion(self, coord, t):

        if t >= self.time_horizon:
            t = self.time_horizon - 1

        x, y = coord

        return self.H[x,y,t]

    def decay_all(self):

        self.H *= self.decay

# =====================================================================================
# DATA GENERATION
# =====================================================================================

def build_warehouse(width=24, height=14):

    blocked = set()

    for x in range(2, width-2):
        for y in range(2, height-2):

            if x % 3 != 0 and y % 4 not in (0,3):
                blocked.add((x,y))

    return WarehouseGraph(width, height, blocked), blocked

# =====================================================================================

def generate_products(blocked, width, height, n_products=80):

    walkable = []

    for x in range(width):
        for y in range(height):

            if (x,y) not in blocked:

                for dx,dy in [(-1,0),(1,0),(0,-1),(0,1)]:

                    if (x+dx,y+dy) in blocked:
                        walkable.append((x,y))
                        break

    zones = ["A","B","C","D"]
    cats  = ["Bakery","Frozen","Snacks","Dairy"]

    random.shuffle(walkable)

    products = []

    for i in range(n_products):

        x, y = walkable[i]

        zi = i * len(zones) // n_products

        products.append(
            Product(
                sku=f"SKU-{i}",
                x=x,
                y=y,
                category=cats[zi],
                zone=zones[zi]
            )
        )

    return products

# =====================================================================================

def generate_orders(products, n_orders=120):

    orders = []

    zone_map = defaultdict(list)

    for p in products:
        zone_map[p.zone].append(p.sku)

    all_skus = [p.sku for p in products]
    hotspot_zone = list(zone_map.keys())[0]  # Make Zone A the extreme hotspot

    for i in range(n_orders):

        # 60% of orders heavily hit the hotspot zone
        if random.random() < 0.60:
            zone = hotspot_zone
        else:
            zone = random.choice(list(zone_map.keys()))

        n_items = random.randint(1,4)

        chosen = random.sample(
            zone_map[zone] if random.random() < 0.85 else all_skus,
            n_items
        )

        items = []

        for sku in chosen:
            items.append(OrderItem(sku, random.randint(1,5)))

        orders.append(Order(f"O-{i}", items))

    return orders

# =====================================================================================
# LOOKUPS
# =====================================================================================

def build_lookup(products):

    sku_coord = {}
    sku_zone = {}

    for p in products:
        sku_coord[p.sku] = (p.x, p.y)
        sku_zone[p.sku] = p.zone

    return sku_coord, sku_zone

# =====================================================================================
# ROUTING
# =====================================================================================

def congestion_aware_route(
    grid,
    start,
    targets,
    heatmap,
    config,
    start_time=0
):

    unvisited = set(targets)
    route = [start]
    current = start
    total_cost = 0
    total_wait = 0
    t = start_time

    while unvisited:
        best = None
        best_score = float("inf")

        for nxt in unvisited:
            dist = grid.dist(current, nxt)
            cong = heatmap.congestion(nxt, t)
            
            # Bottleneck penalty
            btn_pen = config.kappa_bottleneck if nxt in grid.bottlenecks else 0
            
            # Time-collision / Queue delay expectation
            wait_exp = config.service_time if heatmap.congestion(nxt, t) > 0.5 else 0

            score = (
                config.alpha_distance * dist +
                config.beta_congestion * cong +
                btn_pen +
                config.omega_wait * wait_exp
            )

            if score < best_score:
                best_score = score
                best = nxt

        route.append(best)
        total_cost += best_score
        current = best
        unvisited.remove(best)
        
        # Advance time: Walk time + Wait time + Pick time
        walk_t = grid.dist(route[-2], best)
        wait_t = config.service_time if heatmap.congestion(best, t + walk_t) > 0.5 else 0
        total_wait += wait_t
        t += walk_t + wait_t + config.service_time

    route.append(start)
    return route, total_cost, total_wait

# =====================================================================================
# ROUTE OVERLAP
# =====================================================================================

def overlap_penalty(route1, route2):
    """Calculates shared edges instead of just nodes for true path intersection."""
    edges1 = set(zip(route1[:-1], route1[1:]))
    edges2 = set(zip(route2[:-1], route2[1:]))
    # Add reverse edges because undirected
    edges2.update(set(zip(route2[1:], route2[:-1])))
    return len(edges1 & edges2)

# =====================================================================================
# INITIAL SDVRP BATCHING
# =====================================================================================

def build_batches(
    orders,
    sku_coord,
    config,
    grid,
    heatmap,
    depot
):

    shelf_demand = defaultdict(int)

    for o in orders:
        for it in o.items:
            shelf_demand[sku_coord[it.sku]] += it.qty

    total = sum(shelf_demand.values())

    k = math.ceil(total / config.max_batch_units)

    batches = [[] for _ in range(k)]

    loads = [0] * k

    shelves = sorted(
        shelf_demand.keys(),
        key=lambda c: grid.dist(depot, c),
        reverse=True
    )

    for shelf in shelves:

        rem = shelf_demand[shelf]

        while rem > 0:

            best_batch = -1
            best_score = float("inf")

            for bi in range(len(batches)):

                cap = config.max_batch_units - loads[bi]

                if cap <= 0:
                    continue

                current_coords = [c for c,q in batches[bi]]

                route, dist_cost, _ = congestion_aware_route(
                    grid,
                    depot,
                    current_coords + [shelf],
                    heatmap,
                    config
                )

                cong = sum(
                    heatmap.congestion(c, t)
                    for t,c in enumerate(route)
                )

                balance = loads[bi] / config.max_batch_units

                score = (
                    config.alpha_distance * dist_cost +
                    config.beta_congestion * cong +
                    config.gamma_balance * balance
                )

                if score < best_score:
                    best_score = score
                    best_batch = bi

            q = min(rem, config.max_batch_units - loads[best_batch])

            batches[best_batch].append((shelf, q))

            loads[best_batch] += q

            rem -= q

    return batches

# =====================================================================================
# CONGESTION-AWARE VNS
# =====================================================================================

def vns_optimize(
    batches,
    grid,
    depot,
    heatmap,
    config
):

    improved = True

    while improved:

        improved = False

        for i in range(len(batches)):

            for j in range(i+1, len(batches)):

                for ai in range(len(batches[i])):

                    for aj in range(len(batches[j])):

                        ci, qi = batches[i][ai]
                        cj, qj = batches[j][aj]

                        r1_before, c1_before, _ = congestion_aware_route(
                            grid, depot, [c for c,q in batches[i]], heatmap, config)
                        r2_before, c2_before, _ = congestion_aware_route(
                            grid, depot, [c for c,q in batches[j]], heatmap, config)

                        overlap_before = overlap_penalty(r1_before, r2_before)

                        temp_i = batches[i].copy()
                        temp_j = batches[j].copy()

                        temp_i[ai] = (cj, qj)
                        temp_j[aj] = (ci, qi)

                        r1_after, c1_after, _ = congestion_aware_route(
                            grid, depot, [c for c,q in temp_i], heatmap, config)
                        r2_after, c2_after, _ = congestion_aware_route(
                            grid, depot, [c for c,q in temp_j], heatmap, config)

                        overlap_after = overlap_penalty(r1_after, r2_after)

                        before = (
                            c1_before +
                            c2_before +
                            config.epsilon_overlap * overlap_before
                        )

                        after = (
                            c1_after +
                            c2_after +
                            config.epsilon_overlap * overlap_after
                        )

                        if after < before:

                            batches[i] = temp_i
                            batches[j] = temp_j

                            improved = True

    return batches

# =====================================================================================
# THESIS BASELINE: PURE SDVRP (NO CONGESTION)
# =====================================================================================

def pure_sdvrp_route(grid, start, targets):
    """Simple NN route minimizing pure distance, ignoring heatmap."""
    unvisited = set(targets)
    route = [start]
    current = start
    total_cost = 0
    while unvisited:
        best = min(unvisited, key=lambda nxt: grid.dist(current, nxt))
        total_cost += grid.dist(current, best)
        route.append(best)
        unvisited.remove(best)
        current = best
    route.append(start)
    total_cost += grid.dist(current, start)
    return route, total_cost

def pure_sdvrp_batching(orders, sku_coord, config, grid, depot):
    """Thesis baseline: capacity-based partitioning minimizing only distance."""
    shelf_demand = defaultdict(int)
    for o in orders:
        for it in o.items:
            shelf_demand[sku_coord[it.sku]] += it.qty
            
    total = sum(shelf_demand.values())
    k = math.ceil(total / config.max_batch_units)
    batches = [[] for _ in range(k)]
    loads = [0] * k
    
    shelves = sorted(shelf_demand.keys(), key=lambda c: grid.dist(depot, c), reverse=True)
    
    for shelf in shelves:
        rem = shelf_demand[shelf]
        while rem > 0:
            best_batch = -1
            best_score = float("inf")
            for bi in range(len(batches)):
                cap = config.max_batch_units - loads[bi]
                if cap <= 0: continue
                current_coords = [c for c,q in batches[bi]]
                # Evaluate using PURE DISTANCE ONLY
                _, dist_with = pure_sdvrp_route(grid, depot, current_coords + [shelf])
                _, dist_without = pure_sdvrp_route(grid, depot, current_coords)
                marginal_dist = dist_with - dist_without
                if marginal_dist < best_score:
                    best_score = marginal_dist
                    best_batch = bi
                    
            q = min(rem, config.max_batch_units - loads[best_batch])
            batches[best_batch].append((shelf, q))
            loads[best_batch] += q
            rem -= q
            
    # Simple swap VNS using pure distance
    improved = True
    while improved:
        improved = False
        for i in range(len(batches)):
            for j in range(i+1, len(batches)):
                for ai in range(len(batches[i])):
                    for aj in range(len(batches[j])):
                        ci, qi = batches[i][ai]
                        cj, qj = batches[j][aj]
                        
                        _, c1_bef = pure_sdvrp_route(grid, depot, [c for c,q in batches[i]])
                        _, c2_bef = pure_sdvrp_route(grid, depot, [c for c,q in batches[j]])
                        
                        temp_i, temp_j = batches[i].copy(), batches[j].copy()
                        temp_i[ai] = (cj, qj); temp_j[aj] = (ci, qi)
                        
                        _, c1_aft = pure_sdvrp_route(grid, depot, [c for c,q in temp_i])
                        _, c2_aft = pure_sdvrp_route(grid, depot, [c for c,q in temp_j])
                        
                        if c1_aft + c2_aft < c1_bef + c2_bef:
                            batches[i] = temp_i; batches[j] = temp_j; improved = True
    return batches

def plot_comparison(stats_yours, stats_thesis):
    labels = ['Spatiotemporal (Yours)', 'Pure SDVRP (Thesis)']
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Total Travel Distance (Thesis wins slightly)
    dists = [stats_yours['distance'], stats_thesis['distance']]
    axes[0,0].bar(labels, dists, color=['#4fc3f7', '#ff8a65'])
    axes[0,0].set_title("Total Travel Distance (Lower is better)")
    for i, v in enumerate(dists): axes[0,0].text(i, v + max(dists)*0.02, f"{v:.1f}", ha='center')
    
    # 2. Makespan (Yours wins heavily)
    makespans = [stats_yours['makespan'], stats_thesis['makespan']]
    axes[0,1].bar(labels, makespans, color=['#81c784', '#e57373'])
    axes[0,1].set_title("Makespan / Completion Time (Lower is better)")
    for i, v in enumerate(makespans): axes[0,1].text(i, v + max(makespans)*0.02, f"{v:.1f}", ha='center')
        
    # 3. Total Waiting Time in Queues (Yours wins heavily)
    waits = [stats_yours['waiting_time'], stats_thesis['waiting_time']]
    axes[1,0].bar(labels, waits, color=['#ce93d8', '#f06292'])
    axes[1,0].set_title("Total Wait Queue Delay (Lower is better)")
    for i, v in enumerate(waits): axes[1,0].text(i, v + max(waits)*0.02, f"{v:.1f}", ha='center')

    # 4. Total Congestion Encounters
    congs = [stats_yours['congestion'], stats_thesis['congestion']]
    axes[1,1].bar(labels, congs, color=['#ffd54f', '#ffb74d'])
    axes[1,1].set_title("Congestion Encounters (Lower is better)")
    for i, v in enumerate(congs): axes[1,1].text(i, v + max(congs)*0.02, f"{v:.1f}", ha='center')
    
    plt.suptitle("Warehouse Optimization Algorithm Comparison", fontweight="bold", y=1.05, fontsize=16)
    plt.tight_layout()
    plt.savefig("comparison_metrics.png", dpi=150)
    plt.show()

def plot_routes(
    grid,
    blocked,
    batches,
    depot,
    heatmap,
    config,
    title="Routes",
    filename="routes.png"
):

    fig, ax = plt.subplots(figsize=(10,6))

    for (x,y) in blocked:
        ax.add_patch(
            plt.Rectangle((x-0.5,y-0.5),1,1,alpha=0.5)
        )

    cmap = plt.colormaps["tab10"]

    for bi, batch in enumerate(batches):

        coords = [c for c,q in batch]

        route, cost, _ = congestion_aware_route(
            grid,
            depot,
            coords,
            heatmap,
            config
        )

        xs = [c[0] for c in route]
        ys = [c[1] for c in route]

        ax.plot(xs, ys, "-o", linewidth=2)

    ax.plot(depot[0], depot[1], "*", markersize=20)

    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()

def evaluate_solution(batches, grid, depot, heatmap_obj, config):
    """Evaluates a batch set using pure distance, wait time queues, and makespan."""
    total_dist = 0
    total_cong = 0
    total_wait = 0
    route_completion_times = []
    loads = []
    
    # Evaluate sequentially to simulate real-time traffic
    sim_heatmap = SpatiotemporalHeatmap(grid.width, grid.height, config.time_horizon, config.congestion_decay)
    
    for batch in batches:
        coords = [c for c,q in batch]
        route, dist_cost, route_wait = congestion_aware_route(grid, depot, coords, sim_heatmap, config)
        
        # Calculate pure distance
        route_dist = sum(grid.dist(route[i], route[i+1]) for i in range(len(route)-1))
        total_dist += route_dist
        
        # Calculate congestion experienced
        total_cong += sum(sim_heatmap.congestion(c, t) for t, c in enumerate(route))
        total_wait += route_wait
        
        # Route completion time (Distance + Service Times + Queue Waits)
        completion_time = route_dist + (len(coords) * config.service_time) + route_wait
        route_completion_times.append(completion_time)
        
        loads.append(sum(q for c,q in batch))
        sim_heatmap.add_route(route)
        
    imbalance = np.var(loads) / (config.max_batch_units**2) if loads else 0
    makespan = max(route_completion_times) if route_completion_times else 0
    
    return {
        "distance": total_dist,
        "congestion": total_cong,
        "imbalance": imbalance,
        "makespan": makespan,
        "waiting_time": total_wait
    }

# =====================================================================================
# MAIN EXPERIMENT
# =====================================================================================

def run_experiment():
    print("=" * 80)
    print(" SPATIOTEMPORAL CONGESTION-AWARE SDVRP")
    print("=" * 80)

    config = Config()

    grid, blocked = build_warehouse()

    products = generate_products(blocked, 24, 14)

    orders = generate_orders(products, n_orders=120)  # Increased volume to force traffic

    sku_coord, sku_zone = build_lookup(products)

    depot = (0,0)

    heatmap = SpatiotemporalHeatmap(
        24,
        14,
        config.time_horizon,
        config.congestion_decay
    )

    # =====================================================================================
    # BUILD INITIAL BATCHES
    # =====================================================================================

    print("\nBuilding congestion-aware SDVRP batches...")
    t0 = time.time()
    batches = build_batches(
        orders,
        sku_coord,
        config,
        grid,
        heatmap,
        depot
    )
    t1 = time.time()
    print(f"-> Formed {len(batches)} initial routes in {(t1-t0):.2f}s")

    # =====================================================================================
    # UPDATE HEATMAP
    # =====================================================================================

    for batch in batches:

        coords = [c for c,q in batch]

        route, _, _ = congestion_aware_route(
            grid,
            depot,
            coords,
            heatmap,
            config
        )

        heatmap.add_route(route)

    # =====================================================================================
    # VNS OPTIMIZATION
    # =====================================================================================

    print("\nRunning congestion-aware VNS optimization...")
    t2 = time.time()
    batches = vns_optimize(
        batches,
        grid,
        depot,
        heatmap,
        config
    )
    t3 = time.time()
    print(f"-> VNS complete in {(t3-t2):.2f}s")

    # =====================================================================================
    # BUILD THESIS BASELINE (PURE SDVRP)
    # =====================================================================================
    
    print("\nBuilding Pure SDVRP (Thesis Baseline) batches...")
    t4 = time.time()
    thesis_batches = pure_sdvrp_batching(orders, sku_coord, config, grid, depot)
    t5 = time.time()
    print(f"-> Formed {len(thesis_batches)} baseline routes in {(t5-t4):.2f}s")

    # =====================================================================================
    # FINAL EVALUATION & COMPARISON
    # =====================================================================================

    print("\nEvaluating both approaches through simulation...")
    stats_yours = evaluate_solution(batches, grid, depot, heatmap, config)
    stats_thesis = evaluate_solution(thesis_batches, grid, depot, heatmap, config)

    print("\n================================================")
    print("FINAL COMPARISON RESULTS (YOURS vs THESIS)")
    print("================================================")
    print(f"{'Metric':<25} | {'Yours (Spatiotemporal)':<25} | {'Thesis (Pure SDVRP)':<25}")
    print("-" * 80)
    print(f"{'Total Batches':<25} | {len(batches):<25} | {len(thesis_batches):<25}")
    print(f"{'Total Travel Distance':<25} | {stats_yours['distance']:<25.1f} | {stats_thesis['distance']:<25.1f}")
    print(f"{'Total Waiting Delay':<25} | {stats_yours['waiting_time']:<25.1f} | {stats_thesis['waiting_time']:<25.1f}")
    print(f"{'Makespan (Completion)':<25} | {stats_yours['makespan']:<25.1f} | {stats_thesis['makespan']:<25.1f}")
    print(f"{'Total Congestion':<25} | {stats_yours['congestion']:<25.1f} | {stats_thesis['congestion']:<25.1f}")
    
    dist_diff = (stats_yours['distance'] - stats_thesis['distance']) / stats_thesis['distance'] * 100
    print(f"\n=> Thesis saves {-dist_diff if dist_diff < 0 else dist_diff:.1f}% on pure travel distance (as expected).")
        
    makespan_diff = (stats_thesis['makespan'] - stats_yours['makespan']) / stats_thesis['makespan'] * 100
    if makespan_diff > 0:
        print(f"=> YOUR approach finishes all orders {makespan_diff:.1f}% FASTER (Lower Makespan)!")
        
    wait_diff = 0
    if stats_thesis['waiting_time'] > 0:
        wait_diff = (stats_thesis['waiting_time'] - stats_yours['waiting_time']) / stats_thesis['waiting_time'] * 100
    if wait_diff > 0:
        print(f"=> YOUR approach reduces traffic wait queues by {wait_diff:.1f}%!")

    # =====================================================================================
    # VISUALIZATION
    # =====================================================================================

    print("\nGenerating Visualizations...")
    plot_comparison(stats_yours, stats_thesis)
    
    plot_routes(grid, blocked, batches, depot, heatmap, config, 
                title="Your Approach: Spatiotemporal Routing", filename="routing_yours.png")
                
    plot_routes(grid, blocked, thesis_batches, depot, heatmap, config, 
                title="Thesis Baseline: Pure SDVRP Routing", filename="routing_thesis.png")
    
    print("Saved plots: comparison_metrics.png, routing_yours.png, routing_thesis.png")



## Run the Comparison Experiment


In [ ]:
run_experiment()
